In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
df = pd.read_csv('Lab1.csv')
print(df.head())

X = df.drop('churn', axis=1)
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("ข้อมูลตัวอย่าง:")
print(X_train.head())

In [ ]:
# ================= =========================================
# Task 1 & 2: Preprocessing Design & Pipeline Construction
# ==========================================================

# 1. ระบุประเภทของ Feature
num_cols = ['tenure', 'monthly_charges']
cat_cols = ['contract_type', 'payment_method']

# 2. สร้าง Sub-pipeline สำหรับ Numerical Features
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 3. สร้าง Sub-pipeline สำหรับ Categorical Features
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 4. รวม Sub-pipelines ด้วย ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# 5. ฟังก์ชันสร้าง Helper สำหรับต่อ Pipeline กับ Classifier ต่างๆ
def build_full_pipeline(model):
    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('feature_select', SelectKBest(score_func=f_classif, k=4)),
        ('model', model)
    ])

In [ ]:
# ==========================================================
# Task 3: Cross-Model Comparison & Evaluation
# ==========================================================

# สร้าง Pipeline สำหรับ Logistic Regression และ Random Forest
lr_pipeline = build_full_pipeline(LogisticRegression(random_state=42))
rf_pipeline = build_full_pipeline(RandomForestClassifier(random_state=42))

# เทรน Logistic Regression Pipeline
lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)

# เทรน Random Forest Pipeline
rf_pipeline.fit(X_train, y_train)
rf_preds = rf_pipeline.predict(X_test)

In [ ]:
# ---------------------------------------------------------
# แสดงผลการเปรียบเทียบ
# ---------------------------------------------------------
print("=========================================")
print(f"1. Logistic Regression Accuracy: {accuracy_score(y_test, lr_preds):.4f}")
print("=========================================")
print(classification_report(y_test, lr_preds))

print("=========================================")
print(f"2. Random Forest Accuracy      : {accuracy_score(y_test, rf_preds):.4f}")
print("=========================================")
print(classification_report(y_test, rf_preds))